# 01 — Data Generation

## Purpose
Generate a synthetic resume dataset with realistic correlations between latent age group and observable resume features, then persist the dataset and generation metadata to disk for reproducible experiments.

## Outputs
- `data/synthetic_resumes_full.parquet`
- `data/synthetic_resumes_sample.csv`
- `data/generation_metadata.json`

## Notes
- Age (or age_group) is generated as a hidden/protected attribute for analysis only.
- Training features will exclude age fields by design.
- Bias in the simulated callback label is adjustable via a configuration parameter.

## Imports and Configuration

In [1]:
import sys
from pathlib import Path

# Add project root to Python path
PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

print("Project root added to path:", PROJECT_ROOT)

Project root added to path: C:\Users\micro\Documents\Education\ucbai\ucbai-cs-resumefilter


In [2]:
from src.data_generation import (
    GenerationConfig,
    generate_synthetic_resumes,
    save_artifacts
)

## Resume Feature Schema and Controlled Vocabularies

In [5]:
from src.data_generation import RESUME_SCHEMA_DTYPES, DEFAULT_CATEGORY_LEVELS

print("Columns:", list(RESUME_SCHEMA_DTYPES.keys()))
print("Role levels:", DEFAULT_CATEGORY_LEVELS["target_role_level"])

Columns: ['candidate_id', 'application_year', 'target_role_family', 'target_role_level', 'region', 'highest_degree', 'graduation_year', 'school_tier', 'gpa_bucket', 'years_experience_total', 'years_experience_relevant', 'num_employers', 'avg_tenure_years', 'months_since_last_role', 'num_gaps_over_6mo', 'most_recent_title', 'most_recent_company_size', 'management_years', 'reports_max', 'num_skills_listed', 'num_programming_languages', 'num_cloud_platforms', 'num_databases', 'skill_python', 'skill_java', 'skill_javascript', 'skill_go', 'skill_kubernetes', 'skill_aws', 'skill_gcp', 'skill_azure', 'skill_sql', 'skill_spark', 'skill_terraform', 'skill_linux', 'skill_ml', 'legacy_tech_count', 'modern_tech_count', 'cert_count', 'has_top_cloud_cert', 'github_url_present', 'portfolio_url_present', 'open_source_mentions', 'patent_count', 'resume_word_count', 'bullet_count', 'quantified_impact_count', 'keyword_match_score', 'format_clean_score', 'salary_expectation_usd', 'willing_to_relocate', 'r

## Synthetic Data Generator

In [6]:
from src.data_generation import generate_synthetic_resumes
help(generate_synthetic_resumes)

Help on function generate_synthetic_resumes in module src.data_generation:

generate_synthetic_resumes(config: GenerationConfig) -> pd.DataFrame
    Generate a synthetic resume dataset designed to study proxy-bias leakage.

    - age/age_group is generated for fairness evaluation only (not for model features).
    - graduation_year is included as a configurable strong proxy.
    - callback label includes an adjustable penalty for age >= 40 groups.

    Returns:
        DataFrame with columns matching ALL_DTYPES (where applicable).



## Generate Dataset

In [4]:
config = GenerationConfig(
    n_samples=10_000,
    bias_strength=0.10,
    seed=42,
    application_year=2025
)

df = generate_synthetic_resumes(config)

df.head()

,candidate_id,application_year,target_role_family,target_role_level,region,age_group,true_age,highest_degree,school_tier,graduation_year,...,salary_expectation_usd,willing_to_relocate,remote_only,estimated_start_year,tech_recency_score,leadership_signal_score,stability_score,callback,interview,offer
0,cand_000000,2025,SWE,Senior,US-West,30-39,31,None,4,2016,...,138209,False,False,2014,0.419947,0.052986,0.318828,False,<NA>,<NA>
1,cand_000001,2025,IT,Mid,UK,<30,22,AA,1,2024,...,82164,True,False,2025,0.658348,0.002000,0.439782,False,<NA>,<NA>
2,cand_000002,2025,Security,Staff,US-East,30-39,35,PhD,2,2011,...,190247,False,False,2009,0.619487,0.226814,0.310756,True,<NA>,<NA>
3,cand_000003,2025,PM,Staff,UK,50-59,53,AA,1,1993,...,340773,False,True,1995,0.580027,0.741829,0.243552,True,<NA>,<NA>
4,cand_000004,2025,PM,Mid,EU,30-39,33,HS,3,2013,...,143530,False,False,2014,0.646461,0.009163,-0.055558,False,<NA>,<NA>


## Basic Validation and Sanity Checks

In [7]:
df.shape
df["age_group"].value_counts(normalize=True)
df.groupby("age_group")["callback"].mean()
df.describe()

,application_year,true_age,school_tier,graduation_year,years_experience_total,years_experience_relevant,num_employers,avg_tenure_years,months_since_last_role,num_gaps_over_6mo,...,resume_word_count,bullet_count,quantified_impact_count,keyword_match_score,format_clean_score,salary_expectation_usd,estimated_start_year,tech_recency_score,leadership_signal_score,stability_score
count,10000.0,10000.000000,10000.00000,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,...,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,2025.0,39.268100,3.02250,2007.73910,17.297552,14.702780,4.636200,2.640116,5.261300,0.53110,...,644.184400,27.710800,3.941600,0.717225,0.801597,223037.573600,2007.706600,0.573228,0.359609,0.276148
std,0.0,11.501602,1.41619,11.53914,11.596828,10.014091,3.627887,1.013098,5.873097,0.71266,...,180.920769,11.795935,1.989619,0.151855,0.115036,110737.957561,11.600154,0.092808,0.383358,0.126063
min,2025.0,22.000000,1.00000,1980.00000,0.000000,0.000000,0.000000,0.500000,0.000000,0.00000,...,200.000000,5.000000,0.000000,0.450041,0.600040,50000.000000,1980.000000,0.288355,0.000000,-0.121667
25%,2025.0,30.000000,2.00000,1999.00000,7.576744,6.397899,2.000000,1.939528,0.000000,0.00000,...,523.000000,19.000000,3.000000,0.585386,0.704548,128502.500000,1999.000000,0.508113,0.004000,0.190201
50%,2025.0,38.000000,3.00000,2009.00000,15.999185,13.459467,4.000000,2.636937,4.000000,0.00000,...,643.500000,28.000000,4.000000,0.720699,0.801727,205495.500000,2009.000000,0.572748,0.236938,0.275706
75%,2025.0,48.000000,4.00000,2017.00000,25.654793,21.814720,7.000000,3.324044,9.000000,1.00000,...,767.000000,36.000000,5.000000,0.848778,0.902383,301982.250000,2017.000000,0.634086,0.627238,0.362287
max,2025.0,66.000000,5.00000,2026.00000,45.000000,44.794117,18.000000,6.401647,36.000000,5.00000,...,1366.000000,70.000000,13.000000,0.979948,0.999996,527428.000000,2025.000000,0.874399,1.380000,0.746859


## Save Artifacts (Dataset + Metadata)

In [8]:
save_artifacts(
    df,
    out_dir=PROJECT_ROOT / "data" / "baseline",
    config=config
)